# Module 3 Homework 

## Before running


In [1]:
from pathlib import Path

import gdown
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score
from sklearn.tree import DecisionTreeClassifier

DATA_FILE_ID = '1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5'
DATA_PATH = Path('stocks_df_combined_2026_09_18.parquet.brotli')
if not DATA_PATH.exists():
    gdown.download(
        f'https://drive.google.com/uc?id={DATA_FILE_ID}',
        str(DATA_PATH),
        quiet=False,
    )

df_full = pd.read_parquet(DATA_PATH)
df_full.columns = [str(column).strip() for column in df_full.columns]
df_full.head()

Downloading...
From (original): https://drive.google.com/uc?id=1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5
From (redirected): https://drive.google.com/uc?id=1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5&confirm=t&uuid=51190a1f-8a17-4fc1-9276-80204b2cff72
To: /content/stocks_df_combined_2026_09_18.parquet.brotli
100%|██████████| 144M/144M [00:01<00:00, 117MB/s]  


,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,...,growth_brent_oil_7d,growth_brent_oil_30d,growth_brent_oil_90d,growth_brent_oil_365d,growth_btc_usd_1d,growth_btc_usd_3d,growth_btc_usd_7d,growth_btc_usd_30d,growth_btc_usd_90d,growth_btc_usd_365d
0,0.053745,0.061649,0.053745,0.059014,1.031789e+09,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.059014,0.062176,0.059014,0.061122,3.081600e+08,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.061122,0.062703,0.061122,0.062176,1.331712e+08,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.062176,0.062703,0.060068,0.060595,6.776640e+07,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.060595,0.061122,0.059014,0.059541,4.789440e+07,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
date_column = next(column for column in ('Date', 'date') if column in df_full.columns)
df_full[date_column] = pd.to_datetime(df_full[date_column], errors='coerce')
if date_column != 'Date':
    df_full = df_full.rename(columns={date_column: 'Date'})

df = df_full[df_full['Date'] >= '2000-01-01'].copy()

CATEGORICAL = ['Month', 'Weekday', 'Ticker', 'ticker_type']
CATEGORICAL = [column for column in CATEGORICAL if column in df.columns]
TARGET = 'is_positive_growth_30d_future'
REGRESSION_TARGET = 'growth_future_30d'

if TARGET not in df.columns:
    raise KeyError(f'Missing target {TARGET}. Available columns: {list(df.columns)}')

df.shape, df['Date'].min(), df['Date'].max(), CATEGORICAL

((202292, 203),
 Timestamp('2000-01-03 00:00:00'),
 Timestamp('2026-09-18 00:00:00'),
 ['Month', 'Weekday', 'Ticker', 'ticker_type'])

## Question 1: Month and week-of-month dummies

Create values such as `October_w1`, add them to the categorical features, calculate correlations with the binary target, and report the largest absolute correlation.

In [ ]:
# Month can be stored as a datetime column or as a month number/string.
if pd.api.types.is_datetime64_any_dtype(df['Month']):
    month_name = df['Month'].dt.month_name()
else:
    month_number = pd.to_numeric(df['Month'], errors='coerce')
    month_name = month_number.map(lambda value: pd.Timestamp(
        2000, int(value), 1
    ).month_name() if pd.notna(value) else pd.NA)
excluded_columns = set(CATEGORICAL + [
    'Open', 'High', 'Low', 'Close', 'Adj Close_x', 'Adj Close_y',
    'Volume', 'Year', 'Date', 'index', 'index_x', 'index_y',
    'Quarter', TARGET, REGRESSION_TARGET,
])
feature_columns = [
    column for column in df_model.select_dtypes(include='number').columns
    if column not in excluded_columns
    and 'future' not in column
]
    + ((day_of_month - 1) // 7 + 1).astype('string')
)

dummy_variables = pd.get_dummies(
    df[CATEGORICAL + ['month_wom']],
    dtype='int32',
)
df_model = pd.concat([df, dummy_variables], axis=1)
DUMMIES = dummy_variables.columns.tolist()

numeric_columns = df_model.select_dtypes(include='number').columns.tolist()
correlations = df_model[numeric_columns].corr()[TARGET].dropna()
correlations = correlations[~correlations.index.duplicated(keep='last')]
month_wom_columns = [
    column for column in DUMMIES
    if column.startswith('month_wom_')
]
month_wom_corr = correlations.loc[
    correlations.index.intersection(month_wom_columns)
].to_frame('corr')
month_wom_corr['abs_corr'] = month_wom_corr['corr'].abs()
month_wom_corr = month_wom_corr.sort_values('abs_corr', ascending=False)

if month_wom_corr.empty:
    raise ValueError('No month_wom dummy correlations were calculated.')

print('Dummy variable count:', len(DUMMIES))
print('Most correlated month/week dummy:', month_wom_corr.index[0])
print('Q1 answer:', round(month_wom_corr['abs_corr'].iloc[0], 3))
month_wom_corr.head(10)

Dummy variable count: 98
Most correlated month/week dummy: month_wom_October_w4
Q1 answer: 0.025


,corr,abs_corr
month_wom_October_w4,0.024584,0.024584
month_wom_November_w3,0.023251,0.023251
month_wom_February_w1,-0.020861,0.020861
month_wom_January_w2,-0.019846,0.019846
month_wom_January_w5,-0.019792,0.019792
month_wom_November_w2,0.019103,0.019103
month_wom_January_w3,-0.018537,0.018537
month_wom_February_w2,-0.018340,0.018340
month_wom_January_w4,-0.018265,0.018265
month_wom_October_w3,0.017856,0.017856


In [16]:

min_date = df_model['Date'].min()
max_date = df_model['Date'].max()
train_end = min_date + (max_date - min_date) * 0.70
validation_end = train_end + (max_date - min_date) * 0.15
df_model['split'] = np.select(
    [
        df_model['Date'] <= train_end,
        df_model['Date'] <= validation_end,
    ],
    ['train', 'validation'],
    default='test',
)

# Keep one copy of duplicate columns and give every row a unique index.
df_model = df_model.loc[:, ~df_model.columns.duplicated()].copy()
df_model = df_model.reset_index(drop=True)

feature_columns = [
    column for column in df_model.select_dtypes(include='number').columns
    if column not in {TARGET, REGRESSION_TARGET}
    and 'future' not in column
]
feature_columns = [
    column for column in feature_columns
    if df_model[column].notna().to_numpy().any()
]

train_mask = df_model['split'].eq('train')
validation_mask = df_model['split'].eq('validation')
test_mask = df_model['split'].eq('test')
train_validation_mask = train_mask | validation_mask

def clean_xy(frame, columns):
    clean = frame[columns + [TARGET]].dropna().copy()
    return clean[columns], clean[TARGET].astype(int)

X_train, y_train = clean_xy(df_model.loc[train_mask], feature_columns)
X_validation, y_validation = clean_xy(df_model.loc[validation_mask], feature_columns)
X_test, y_test = clean_xy(df_model.loc[test_mask], feature_columns)
X_train_validation, y_train_validation = clean_xy(
    df_model.loc[train_validation_mask],
    feature_columns,
)
print(df_model['split'].value_counts())

split
train         136842
test           33163
validation     32287
Name: count, dtype: int64


## Question 2: New hand rules

The homework provides these two rules from the decision-tree branches. `pred3` should use DGS10 and DGS5; `pred4` should use DGS10 and FEDFUNDS. Precision is calculated only for positive predictions on the test set.

In [17]:
def add_hand_rules(frame):
    frame = frame.copy()
    frame['pred3'] = (
        (frame['DGS10'] <= 4.5)
        & (frame['DGS5'] <= 4)
    ).astype(int)
    frame['pred4'] = (
        (frame['DGS10'] > 4)
        & (frame['FEDFUNDS'] <= 4.795)
    ).astype(int)
    return frame

for required in ['DGS10', 'DGS5', 'FEDFUNDS']:
    if required not in df_model.columns:
        raise KeyError(f'Missing required rule column: {required}')

df_model = add_hand_rules(df_model)

def positive_precision(frame, prediction_column):
    selected = frame[frame[prediction_column].eq(1)].dropna(subset=[TARGET])
    if selected.empty:
        return np.nan
    return precision_score(selected[TARGET], selected[prediction_column], zero_division=0)

for prediction in ['pred3', 'pred4']:
    test_rules = df_model.loc[test_mask]
    print(prediction, 'positive test predictions:', int(test_rules[prediction].sum()))
    print(prediction, 'test precision:', round(positive_precision(test_rules, prediction), 3))

# Q2 answer is the precision of the rule that produces positive test predictions.
q2_precisions = {prediction: positive_precision(df_model.loc[test_mask], prediction) for prediction in ['pred3', 'pred4']}
q2_precisions

pred3 positive test predictions: 15910
pred3 test precision: 0.588
pred4 positive test predictions: 15921
pred4 test precision: 0.503


{'pred3': 0.5877435575109994, 'pred4': 0.5025438100621821}

In [18]:
# Add the earlier hand rules used for the comparison in Q3.
df_model['pred0'] = (df_model['cci'] > 200).astype(int) if 'cci' in df_model else 0
df_model['pred1'] = (df_model['growth_30d'] > 1).astype(int) if 'growth_30d' in df_model else 0
df_model['pred2'] = (
    (df_model['growth_30d'] > 1)
    & (df_model['growth_snp500_30d'] > 1)
).astype(int) if {'growth_30d', 'growth_snp500_30d'}.issubset(df_model.columns) else 0

hand_predictions = ['pred0', 'pred1', 'pred2', 'pred3', 'pred4']


## Question 3: Unique correct predictions from a depth-10 tree

Fit a `DecisionTreeClassifier(max_depth=10, random_state=42)` on train plus validation. Then count test records where `pred5_clf_10` is correct and every hand rule is wrong.

In [19]:
clf10 = DecisionTreeClassifier(max_depth=10, random_state=42)
clf10.fit(X_train_validation, y_train_validation)

all_clean = df_model[feature_columns + [TARGET]].dropna().copy()
df_model.loc[all_clean.index, 'pred5_clf_10'] = clf10.predict(all_clean[feature_columns])
df_model['pred5_clf_10'] = df_model['pred5_clf_10'].fillna(0).astype(int)

all_predictions_correct = pd.DataFrame({
    prediction: df_model[prediction].eq(df_model[TARGET])
    for prediction in hand_predictions + ['pred5_clf_10']
})
df_model['only_pred5_is_correct'] = (
    all_predictions_correct['pred5_clf_10']
    & ~all_predictions_correct[hand_predictions].any(axis=1)
).astype(int)

q3_answer = int(
    df_model.loc[test_mask, 'only_pred5_is_correct'].sum()
)
print('Q3 answer:', q3_answer)

Q3 answer: 1187


In [20]:
def tree_precision(max_depth):
    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    model.fit(X_train_validation, y_train_validation)
    predictions = model.predict(X_test)
    return model, precision_score(y_test, predictions, zero_division=0)

depth_results = []
for max_depth in range(1, 13):
    model, precision = tree_precision(max_depth)
    depth_results.append({
        'max_depth': max_depth,
        'test_precision': precision,
    })

depth_results = pd.DataFrame(depth_results)
best_max_depth = int(depth_results.loc[depth_results['test_precision'].idxmax(), 'max_depth'])
print('Q4 answer:', best_max_depth)
depth_results

Q4 answer: 4


,max_depth,test_precision
0,1,0.595329
1,2,0.595329
2,3,0.595329
3,4,0.595516
4,5,0.589422
5,6,0.584163
6,7,0.565395
7,8,0.573831
8,9,0.573959
9,10,0.589178


## Question 5: Exploratory - what data is missing?

Possible additions include analyst estimates, earnings revisions, short interest, options-implied volatility, liquidity/spread data, sector returns, market breadth, and sentiment.